# 1. Optimal Simulation Parameter Analysis (Density-Scaled)

Determines optimal simulation discretization parameters via empirical stability sweeps.
- **Goal:** Find the minimum passenger density (pax/route/hr) required for routing optimization to reach statistical steady-state (CV ≤ 5%).
- **Phase 1 (SPT Sweep):** Maximize SPT while keeping CV ≤ 5% (fixed 120 min simulated duration).
- **Phase 2 (Bisection):** Minimize tick count via bisection (max 10 hours simulated). Includes a circuit breaker for inherent variance flatlines.

**Output:** `rnd/csv/rnd1_stability_results.csv`

In [ ]:
import os, sys, time, gc, random
import numpy as np
import pandas as pd
import ast
import re
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.getcwd())
random.seed(42)
np.random.seed(42)

from utils_simplified import (
    reuse_citygraph, reuse_ddm, generate_route_system,
    generate_dummy_yaml, SimEnvironment, run_simulations_parallel
)

cg_pkl = "rnd/pkl/profile_p1.pkl"
ddm_pkl = "rnd/pkl/ddm_8am.pkl"

print("Loading cached CityGraph and DDM...")
t0 = time.time()
G_city = reuse_citygraph(cg_pkl)
ddm = reuse_ddm(ddm_pkl)
print(f"Loaded in {time.time()-t0:.1f}s | {len(G_city.nodes)} nodes, {len(G_city.graph)} edges")

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================

ROUTE_COUNTS     = [3, 6, 10, 14, 18]    
# Density: Off-peak, Standard, Rush Hour
PAX_PER_ROUTE_HR = [50, 150, 300]      
REPLICATIONS     = 7                     
JEEPS_PER_ROUTE  = 10

# Tightened SPT sweep to maintain vehicle physics integrity
SPT_VALUES       = [10, 20, 30] 
# 2 hours simulated for Phase 1 baseline
SWEEP_SIM_S      = 7200       
# Allows up to 10 hours simulated to find stability
BISECT_BOUNDS    = (100, 3600) 

CV_THRESHOLD = 0.05
os.makedirs("rnd/csv", exist_ok=True)
os.makedirs("rnd/configs", exist_ok=True)
os.makedirs("rnd/images", exist_ok=True)

print(f"Route counts: {ROUTE_COUNTS}")
print(f"Densities tested: {PAX_PER_ROUTE_HR} pax/route/hr")
print(f"Replications: {REPLICATIONS} (run in parallel)")
print(f"SPT sweep:    {SPT_VALUES} (fixed {SWEEP_SIM_S}s simulated)")
print(f"Bisection:    {BISECT_BOUNDS} (dynamic while loop with circuit breaker)")

In [ ]:
# ==========================================
# FUNCTION DEFINITIONS (Defined only once!)
# ==========================================

def compute_cv(scores):
    if len(scores) < 2:
        return 1.0
    arr = np.array(scores, dtype=float)
    m = arr.mean()
    return (arr.std(ddof=1) / m) if m > 1e-9 else 1.0

class MockJS:
    def __init__(self, routes):
        self.routes = routes

def evaluate_cv_parallel(routes, spt, num_ticks, total_pass_rate):
    n_jeeps = len(routes) * JEEPS_PER_ROUTE
    yaml_path = f"rnd/configs/_rnd1_s{spt}_t{num_ticks}_r{len(routes)}_p{int(total_pass_rate)}.yaml"
    
    generate_dummy_yaml(
        yaml_path,
        **{
            "simulation.seconds_per_tick":        spt,
            "simulation.num_ticks":               num_ticks,
            "simulation.spawn_rate_per_hour":     total_pass_rate,
            "simulation.total_allocatable_jeeps": n_jeeps,
            "cg_pkl": cg_pkl,
            "ddm_pkl": ddm_pkl,
        }
    )

    envs = [
        SimEnvironment(
            tg=None, yaml_file=yaml_path,
            jeep_system=MockJS(routes), sampler=ddm,
            delete_yaml_when_done=(i == REPLICATIONS - 1)
        )
        for i in range(REPLICATIONS)
    ]

    results = run_simulations_parallel(envs)
    scores = [r.score for r in results if r is not None]
    cv = compute_cv(scores)

    del envs, results
    gc.collect()
    return cv, np.mean(scores) if scores else float('nan'), scores

In [ ]:
# ==========================================
# MAIN EXECUTION LOOP
# ==========================================

all_results = []
total_scenarios = len(ROUTE_COUNTS) * len(PAX_PER_ROUTE_HR)
scenario_i = 0

for n_routes in ROUTE_COUNTS:
    random.seed(42 + n_routes)
    np.random.seed(42 + n_routes)
    print(f"\n{'='*70}")
    print(f"GENERATING {n_routes} ROUTES...")
    t0 = time.time()
    routes = generate_route_system(n_routes, G_city, ddm)
    print(f"  {n_routes} routes, {sum(len(r.path) for r in routes)} edges, "
          f"{n_routes*JEEPS_PER_ROUTE} jeeps | {time.time()-t0:.1f}s")

    for pax_density in PAX_PER_ROUTE_HR:
        scenario_i += 1
        
        # DYNAMIC SCALING: Calculate total map spawn rate
        total_pass_rate = pax_density * n_routes
        
        print(f"\n{'─'*60}")
        print(f"SCENARIO {scenario_i}/{total_scenarios}: R={n_routes}, Density={pax_density}/route/hr (Total: {total_pass_rate}/hr)")

        # ── Phase 1: SPT Sweep ──
        print(f"  [SPT Sweep] fixed_sim={SWEEP_SIM_S}s")
        optimal_spt = SPT_VALUES[0]
        spt_log = []

        for spt in SPT_VALUES:
            num_ticks = max(10, SWEEP_SIM_S // spt)
            t0 = time.time()
            cv, mean_fit, scores = evaluate_cv_parallel(routes, spt, num_ticks, total_pass_rate)
            wall = time.time() - t0
            status = 'PASS' if cv <= CV_THRESHOLD else 'FAIL'
            print(f"    SPT={spt:3d}s ticks={num_ticks:4d} | "
                  f"CV={cv:.4f} mean={mean_fit:.1f} | {wall:.0f}s | {status}")
            spt_log.append({'spt': spt, 'ticks': num_ticks, 'cv': cv})

            if cv <= CV_THRESHOLD:
                optimal_spt = spt
            else:
                print(f"    -> Early stop")
                break

        print(f"  -> Optimal SPT: {optimal_spt}s")

        # ── Phase 2: True Bisection ──
        print(f"  [Ticks Bisect] spt={optimal_spt}s, range={BISECT_BOUNDS}")
        lo, hi = BISECT_BOUNDS
        optimal_ticks = hi
        ticks_log = []
        cv_history = []
        bi = 0

        while lo <= hi:
            bi += 1
            mid = (lo + hi) // 2
            t0 = time.time()
            
            # The execution call
            cv, mean_fit, scores = evaluate_cv_parallel(routes, optimal_spt, mid, total_pass_rate)
            
            wall = time.time() - t0
            status = 'PASS' if cv <= CV_THRESHOLD else 'FAIL'
            print(f"    iter {bi}: ticks={mid:4d} | "
                  f"CV={cv:.4f} mean={mean_fit:.1f} | {wall:.0f}s | {status}")
            ticks_log.append({'ticks': mid, 'cv': cv})
            cv_history.append(cv)

            # CIRCUIT BREAKER: If CV changes by less than 0.001 over 3 jumps, it's flatlined.
            if len(cv_history) >= 3:
                recent = cv_history[-3:]
                if max(recent) - min(recent) < 0.001 and cv > CV_THRESHOLD:
                    print("    -> [CIRCUIT BREAKER] Variance flatlined. System inherently unstable at this density.")
                    break

            if cv <= CV_THRESHOLD:
                optimal_ticks = mid
                hi = mid - 1 # Search lower bounds for efficiency
            else:
                lo = mid + 1 # Search higher bounds for stability

        sim_min = (optimal_ticks * optimal_spt) / 60
        print(f"  -> Optimal ticks: {optimal_ticks} ({sim_min:.0f} sim min)")

        all_results.append({
            "num_routes": n_routes, 
            "pax_density": pax_density,
            "total_pass_rate": total_pass_rate,
            "optimal_spt": optimal_spt, 
            "optimal_ticks": optimal_ticks,
            "sim_duration_min": sim_min,
            "spt_log": str(spt_log), 
            "ticks_log": str(ticks_log),
        })
        pd.DataFrame(all_results).to_csv(
            "rnd/csv/rnd1_stability_results_partial.csv", index=False)

    del routes
    gc.collect()

df = pd.DataFrame(all_results)
df.to_csv("rnd/csv/rnd1_stability_results.csv", index=False)
print(f"\n{'='*70}")
print("EXPERIMENT COMPLETE — rnd/csv/rnd1_stability_results.csv")
print(f"{'='*70}")

In [ ]:
# ==========================================
# VISUALIZE CONVERGENCE 
# ==========================================

df = pd.read_csv("rnd/csv/rnd1_stability_results.csv")

rows = []
for index, row in df.iterrows():
    raw_string = str(row['ticks_log'])
    
    # Strip numpy wrapper to allow safe parsing
    clean_string = re.sub(r'np\.float64\((.*?)\)', r'\1', raw_string)
    ticks_log = ast.literal_eval(clean_string)
    
    for log in ticks_log:
        rows.append({
            'num_routes': row['num_routes'],
            'pax_density': row['pax_density'],
            'total_pass_rate': row['total_pass_rate'],
            'ticks': log['ticks'],
            'spt': row['optimal_spt'],
            'sim_duration_min': (log['ticks'] * row['optimal_spt']) / 60,
            'cv': log['cv']
        })

cv_df = pd.DataFrame(rows)

sns.set_theme(style="whitegrid")
g = sns.FacetGrid(cv_df, col="num_routes", hue="pax_density", col_wrap=3, height=4, palette="magma")
g.map(sns.lineplot, "sim_duration_min", "cv", marker="o")
g.map(plt.axhline, y=0.05, color="red", linestyle="--", label="5% Threshold")

g.set_axis_labels("Simulated Duration (Mins)", "Coefficient of Variation (CV)")
g.set_titles("Routes: {col_name}")
g.add_legend(title="Density (Pax/Route/Hr)")
plt.subplots_adjust(top=0.9)
g.fig.suptitle("Optimization Stability by Passenger Density (Target CV ≤ 0.05)", fontweight="bold", fontsize=14)

plt.savefig("rnd/images/rnd1_convergence_density.png", dpi=150, bbox_inches="tight")
plt.show()